In [ ]:
# Count stored detections for each person ID across all videos.
# These counts measure observations, not screen time or importance.
import sqlite3
from contextlib import closing
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

ROOT = next(
    root
    for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "backend" / "analysis_db").is_dir()
)
DB_PATH = ROOT / "data" / "analysis.sqlite"
CUTOFF_PERCENTILE = 99  # Change this value, then run all three cells.

if not 0 <= CUTOFF_PERCENTILE <= 100:
    raise ValueError("CUTOFF_PERCENTILE must be between 0 and 100.")

with closing(
    sqlite3.connect(f"{Path(DB_PATH).resolve().as_uri()}?mode=ro", uri=True)
) as connection:
    person_occurrences = dict(
        connection.execute("""
        SELECT p.person_id, COUNT(d.detection_id) AS occurrences
        FROM Person AS p
        LEFT JOIN FaceDetection AS d ON d.person_id = p.person_id
        GROUP BY p.person_id
        ORDER BY occurrences DESC, p.person_id
    """)
    )

counts = np.fromiter(person_occurrences.values(), dtype=np.int64)
if counts.size == 0:
    raise ValueError("The database has no person IDs to analyze.")
print(f"Person IDs: {counts.size:,} | Stored detections: {counts.sum():,}")


In [ ]:
# Round fractional cutoffs up. Keep all IDs that meet the cutoff, including ties.
table = [
    "| Percentile | Occurrences | Minimum occurrences | IDs kept | IDs kept (%) |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for percentile in sorted({50, 75, 90, CUTOFF_PERCENTILE}):
    value = np.percentile(counts, percentile, method="linear")
    minimum = int(np.ceil(value))
    kept = int(np.count_nonzero(counts >= minimum))
    table.append(
        f"| {percentile:g} | {value:.2f} | {minimum} | {kept:,} | {100 * kept / counts.size:.1f} |"
    )
display(Markdown("\n".join(table)))


In [ ]:
# Recalculate from the current database on each full run. No records are changed.
MIN_OCCURRENCES = int(
    np.ceil(np.percentile(counts, CUTOFF_PERCENTILE, method="linear"))
)
retained_occurrences = {
    person_id: count
    for person_id, count in person_occurrences.items()
    if count >= MIN_OCCURRENCES
}
RETAINED_PERSON_IDS = set(retained_occurrences)

print(f"Selected percentile: {CUTOFF_PERCENTILE:g}")
print(f"Keep IDs with at least {MIN_OCCURRENCES:,} stored detections.")
print(f"Person IDs kept: {len(RETAINED_PERSON_IDS):,} of {counts.size:,}")
print(f"Person IDs excluded: {counts.size - len(RETAINED_PERSON_IDS):,}")
print(
    f"Detections for kept IDs: {sum(retained_occurrences.values()):,} of {counts.sum():,}"
)
